In [ ]:
import pandas as pd
import torch
import torch_geometric
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
import io
from PIL import Image
from collections import defaultdict
from IPython.display import SVG,display
import os
import pickle

def obtain_topk(x,k_perc):
    
    k = int(np.round(len(x)*k_perc))
    
    _, idx = torch.topk(x, k)                
    mask = torch.zeros_like(x, dtype=torch.float)
    mask[idx] = 1.0    

    return mask



In [ ]:


from rdkit import Chem
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score,precision_score,recall_score

def node_attribution_metrics(
    row,
    node_scores,                       # array-like: probabilities or 0/1 per atom
    alert_smarts_dict,
    n_alerts=12,
    binarize_threshold=0.5             # for accuracy only
):
    """
    Compute node attribution accuracy and AUROC for one molecule.

    node_scores can be probabilities in [0,1] or hard 0/1.
    AUROC uses raw scores. Accuracy uses binarized scores.
    If q_vec has only positives or only negatives, AUROC = 0.0.
    """
    # 1) SMILES -> Mol
    smiles = row["smiles"]
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles!r}")
    n_atoms = mol.GetNumAtoms()

    # 2) Active alerts
    alert_cols = [f"Alert {i}" for i in range(1, n_alerts + 1)]
    active_alerts = [c for c in alert_cols if int(row.get(c, 0)) == 1]

    # 3) Union of substructure atom indices (q)
    q_atom_indices = set()
    for alert in active_alerts:
        smarts = alert_smarts_dict.get(alert)
        if not smarts:
            continue
        qmol = Chem.MolFromSmarts(smarts)
        if qmol is None:
            continue
        for match in mol.GetSubstructMatches(qmol):
            q_atom_indices.update(match)

    q_vec = np.zeros(n_atoms, dtype=int)
    if q_atom_indices:
        idxs = np.fromiter(q_atom_indices, dtype=int, count=len(q_atom_indices))
        if (idxs < 0).any() or (idxs >= n_atoms).any():
            raise ValueError("Out-of-range atom index in q.")
        q_vec[idxs] = 1

    # 4) Align node scores
    node_scores = np.asarray(node_scores, dtype=float)
    if node_scores.shape[0] != n_atoms:
        raise ValueError(
            f"node_scores length ({node_scores.shape[0]}) != number of atoms ({n_atoms})"
        )

    # 5) Metrics
    node_scores_bin = (node_scores >= binarize_threshold).astype(int)
    acc = float(accuracy_score(q_vec, node_scores_bin))
    prec = float(precision_score(q_vec, node_scores_bin, zero_division=0))
    rec = float(recall_score(q_vec, node_scores_bin, zero_division=0))

    # AUROC: force defined result
    if len(np.unique(q_vec)) == 1:  
        auroc = 0.0
    else:
        auroc = float(roc_auc_score(q_vec, node_scores))

    tp = int(((q_vec == 1) & (node_scores_bin == 1)).sum())
    tn = int(((q_vec == 0) & (node_scores_bin == 0)).sum())
    fp = int(((q_vec == 0) & (node_scores_bin == 1)).sum())
    fn = int(((q_vec == 1) & (node_scores_bin == 0)).sum())

    return {
        "mol": row['name'],
        "active_alerts": active_alerts,
        "q_atom_indices": set(map(int, q_atom_indices)),
        "q_vec": q_vec,
        "node_scores_raw": node_scores,
        "node_scores_bin": node_scores_bin,
        "accuracy": acc,
        "auroc": auroc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "precision": prec,
        "recall": rec,
    }



In [ ]:

Alert_dict = {'Alert 1':"C12CCCCC1C3C(CCC3)CC2", 
              'Alert 2':"NN", 
              'Alert 3':"a[C!R]C(=O)[OH]",
              'Alert 4':"[#6]S(=O)(=O)N[#6]", 
              'Alert 5':"c1ccccc1[NH2]", 
              'Alert 6':"O = [S;X3]", 
              'Alert 7':"[S;X2&!R]", 
              'Alert 8':"a[C!R](=O)a", 
              'Alert 9':"C[F,Cl,Br,I]", 
              'Alert 10':"C1CC1N", 
              'Alert 11':"[O]c1ccc([N])cc1", 
              'Alert 12':"N1c2ccccc2Sc2ccccc12"}


acc_gnn_encoder_10 = []
acc_gnn_encoder_25 = []
acc_gnn_encoder_50 = []
acc_gnn_encoder_sem_10 = []
acc_gnn_encoder_sem_25 = []
acc_gnn_encoder_sem_50 = []
acc_gnn_encoder_semCL_10 = []
acc_gnn_encoder_semCL_25 = []
acc_gnn_encoder_semCL_50 = []

pre_gnn_encoder_10 = []
pre_gnn_encoder_25 = []
pre_gnn_encoder_50 = []
pre_gnn_encoder_sem_10 = []
pre_gnn_encoder_sem_25 = []
pre_gnn_encoder_sem_50 = []
pre_gnn_encoder_semCL_10 = []
pre_gnn_encoder_semCL_25 = []
pre_gnn_encoder_semCL_50 = []

rec_gnn_encoder_10 = []
rec_gnn_encoder_25 = []
rec_gnn_encoder_50 = []
rec_gnn_encoder_sem_10 = []
rec_gnn_encoder_sem_25 = []
rec_gnn_encoder_sem_50 = []
rec_gnn_encoder_semCL_10 = []
rec_gnn_encoder_semCL_25 = []
rec_gnn_encoder_semCL_50 = []

auc_gnn_encoder_10 = []
auc_gnn_encoder_25 = []
auc_gnn_encoder_50 = []
auc_gnn_encoder_sem_10 = []
auc_gnn_encoder_sem_25 = []
auc_gnn_encoder_sem_50 = []
auc_gnn_encoder_semCL_10 = []
auc_gnn_encoder_semCL_25 = []
auc_gnn_encoder_semCL_50 = []

auc_gnn_encoder = []
auc_gnn_encoder_sem = []
auc_gnn_encoder_semCL = []
comp_name = []
smile_name = []

for assay in ['Liver_2vs']:
    
    datadf = pd.read_excel("./MolCLR/data/Liver/datasets_valid_and_splits/" + assay + "_df.xlsx")
    datadf_pos = datadf[datadf['target'] == 1]

    
    attribution_df = pd.read_excel("./MolCLR/data/Liver/Liver_.xlsx")
    attribution_df['name'] = attribution_df['Drug Name']
    attribution_df['smiles'] = attribution_df['n.sMILES']
    
    datadf_pos = datadf_pos.merge(attribution_df[['name','smiles','attribution','Alert 1', 'Alert 2', 'Alert 3', 'Alert 4', 'Alert 5', 'Alert 6', 'Alert 7', 'Alert 8', 'Alert 9', 'Alert 10', 'Alert 11', 'Alert 12']],on=['name','smiles'])
    
    
    for xplain_methon in ['Layer_Saliency']:
        for index,row in datadf_pos.iterrows():
            
            if row[['Alert 1','Alert 2', 'Alert 3', 'Alert 4', 'Alert 5', 'Alert 6', 
                    'Alert 7','Alert 8', 'Alert 9', 'Alert 10', 'Alert 11', 'Alert 12']].values.sum() != 0:
            
                compund_name = row['name']
                smiles_name = row['smiles']

                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_127 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_128 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_129 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_130 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_131 = pickle.load(fp)

                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_127 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_128 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_129 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_130 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_131 = pickle.load(fp)

                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_127 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_128 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_129 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_130 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_131 = pickle.load(fp)




                ## GNN encoder
                gnn_encoder_mean_node_scores = torch.mean(torch.concat((gnn_encoder_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)



                gnn_encoder_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)


                res_gnn_encoder = {'node_scores':gnn_encoder_mean_node_scores,
                                   'node_scores_10':obtain_topk(gnn_encoder_mean_node_scores,0.10),
                                    'node_scores_25':obtain_topk(gnn_encoder_mean_node_scores,0.25),
                                   'node_scores_50':obtain_topk(gnn_encoder_mean_node_scores,0.50),
                                   'bond_idx':gnn_encoder_explanation_127['bond_idx'],
                                    'edge_scores':gnn_encoder_mean_edge_scores, 
                                   'edge_scores_25':obtain_topk(gnn_encoder_mean_edge_scores,0.25),
                                   'edge_scores_50':obtain_topk(gnn_encoder_mean_edge_scores,0.50)}


                ## GNN encoder + sem
                gnn_encoder_sem_mean_node_scores = torch.mean(torch.concat((gnn_encoder_sem_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)

                gnn_encoder_sem_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_sem_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)


                res_gnn_encoder_sem = {'node_scores':gnn_encoder_sem_mean_node_scores,
                                       'node_scores_10':obtain_topk(gnn_encoder_sem_mean_node_scores,0.10),
                                    'node_scores_25':obtain_topk(gnn_encoder_sem_mean_node_scores,0.25),
                                       'node_scores_50':obtain_topk(gnn_encoder_sem_mean_node_scores,0.50),
                                   'atom_idx_in_hetero':gnn_encoder_sem_explanation_129['atom_idx_in_hetero'],
                                    'edge_scores_25':gnn_encoder_sem_mean_edge_scores,
                                   'edge_scores_25':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.25),
                                    'edge_scores_50':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.50),
                                   'bond_idx_in_hetero':gnn_encoder_sem_explanation_129['bond_idx_in_hetero']}

                ## GNN encoder + sem CL
                gnn_encoder_semCL_mean_node_scores = torch.mean(torch.concat((gnn_encoder_semCL_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)

                gnn_encoder_semCL_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_semCL_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)

                res_gnn_encoder_semCL = {'node_scores':gnn_encoder_semCL_mean_node_scores,
                                         'node_scores_10':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.10),
                                        'node_scores_25':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.25),
                                         'node_scores_50':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.50),
                                   'atom_idx_in_hetero':gnn_encoder_semCL_explanation_127['atom_idx_in_hetero'],
                                'edge_scores':gnn_encoder_sem_mean_edge_scores,
                                   'edge_scores_25':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.25),
                                    'edge_scores_50':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.50),
                                   'bond_idx_in_hetero':gnn_encoder_semCL_explanation_127['bond_idx_in_hetero']}
                
                acc_gnn_encoder_10.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_10'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_25.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_25'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_50.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_50'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_sem_10.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_10'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_sem_25.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_25'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_sem_50.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_50'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_semCL_10.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_10'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_semCL_25.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_25'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_semCL_50.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_50'], Alert_dict, n_alerts=12)['accuracy'])

                pre_gnn_encoder_10.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_10'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_25.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_25'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_50.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_50'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_sem_10.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_10'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_sem_25.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_25'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_sem_50.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_50'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_semCL_10.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_10'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_semCL_25.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_25'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_semCL_50.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_50'], Alert_dict, n_alerts=12)['precision'])

                
                rec_gnn_encoder_10.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_10'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_25.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_25'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_50.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_50'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_sem_10.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_10'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_sem_25.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_25'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_sem_50.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_50'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_semCL_10.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_10'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_semCL_25.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_25'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_semCL_50.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_50'], Alert_dict, n_alerts=12)['recall'])

                
                auc_gnn_encoder_10.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_10'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_25.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_25'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_50.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_50'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_sem_10.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_10'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_sem_25.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_25'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_sem_50.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_50'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_semCL_10.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_10'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_semCL_25.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_25'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_semCL_50.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_50'], Alert_dict, n_alerts=12)['auroc'])

                auc_gnn_encoder.append(node_attribution_metrics(row, res_gnn_encoder['node_scores'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_sem.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_semCL.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores'], Alert_dict, n_alerts=12)['auroc'])
                
                comp_name.append(compund_name)
                smile_name.append(smiles_name)

In [ ]:
df_name_scores_25 = pd.DataFrame({'smiles':smile_name,'name':comp_name,"rec_gnn_encoder_25":rec_gnn_encoder_25,
                              "rec_gnn_encoder_sem_25":rec_gnn_encoder_sem_25,
                              "rec_gnn_encoder_semCL_25":rec_gnn_encoder_semCL_25,
                              "pre_gnn_encoder_25":pre_gnn_encoder_25,
                              "pre_gnn_encoder_sem_25":pre_gnn_encoder_sem_25,
                              "pre_gnn_encoder_semCL_25":pre_gnn_encoder_semCL_25,
                              "auc_gnn_encoder_25":auc_gnn_encoder_25,
                              "auc_gnn_encoder_sem_25":auc_gnn_encoder_sem_25,
                              "auc_gnn_encoder_semCL_25":auc_gnn_encoder_semCL_25})

df_name_scores_50 = pd.DataFrame({'smiles':smile_name,'name':comp_name,
                              "rec_gnn_encoder_50":rec_gnn_encoder_50,
                              "rec_gnn_encoder_sem_50":rec_gnn_encoder_sem_50,
                              "rec_gnn_encoder_semCL_50":rec_gnn_encoder_semCL_50,
                              "pre_gnn_encoder_50":pre_gnn_encoder_50,
                              "pre_gnn_encoder_sem_50":pre_gnn_encoder_sem_50,
                              "pre_gnn_encoder_semCL_50":pre_gnn_encoder_semCL_50,
                              "auc_gnn_encoder_50":auc_gnn_encoder_50,
                              "auc_gnn_encoder_sem_50":auc_gnn_encoder_sem_50,
                              "auc_gnn_encoder_semCL_50":auc_gnn_encoder_semCL_50})

In [ ]:
names_rec_25 = df_name_scores_25[(df_name_scores_25['rec_gnn_encoder_sem_25'] > df_name_scores_25['rec_gnn_encoder_semCL_25']) & 
                 (df_name_scores_25['rec_gnn_encoder_sem_25'] > df_name_scores_25['rec_gnn_encoder_25'])]['name'].values

names_pre_25 = df_name_scores_25[(df_name_scores_25['pre_gnn_encoder_sem_25'] > df_name_scores_25['pre_gnn_encoder_semCL_25']) & 
                 (df_name_scores_25['pre_gnn_encoder_sem_25'] > df_name_scores_25['pre_gnn_encoder_25'])]['name'].values

names_auc_25 = df_name_scores_25[(df_name_scores_25['auc_gnn_encoder_sem_25'] > df_name_scores_25['auc_gnn_encoder_semCL_25']) & 
                 (df_name_scores_25['auc_gnn_encoder_sem_25'] > df_name_scores_25['auc_gnn_encoder_25'])]['name'].values


names_25 = set(names_rec_25).intersection(set(names_pre_25)).intersection(set(names_auc_25))


df_names_25 = df_name_scores_25[df_name_scores_25['name'].isin(names_25)]

df_names_25 = df_names_25.merge(datadf_pos[['Alert 1','Alert 2', 'Alert 3', 'Alert 4', 'Alert 5', 'Alert 6', 
                    'Alert 7','Alert 8', 'Alert 9', 'Alert 10', 'Alert 11', 'Alert 12','name']],on="name")

names_rec_50 = df_name_scores_50[(df_name_scores_50['rec_gnn_encoder_semCL_50'] > df_name_scores_50['rec_gnn_encoder_sem_50']) & 
                 (df_name_scores_50['rec_gnn_encoder_semCL_50'] > df_name_scores_50['rec_gnn_encoder_50'])]['name'].values

names_pre_50 = df_name_scores_50[(df_name_scores_50['pre_gnn_encoder_semCL_50'] > df_name_scores_50['pre_gnn_encoder_sem_50']) & 
                 (df_name_scores_50['pre_gnn_encoder_semCL_50'] > df_name_scores_50['pre_gnn_encoder_50'])]['name'].values

names_auc_50 = df_name_scores_50[(df_name_scores_50['auc_gnn_encoder_semCL_50'] > df_name_scores_50['auc_gnn_encoder_sem_50']) & 
                 (df_name_scores_50['auc_gnn_encoder_semCL_50'] > df_name_scores_50['auc_gnn_encoder_50'])]['name'].values


names_50 = set(names_rec_50).intersection(set(names_pre_50)).intersection(set(names_auc_50))



df_names_50 = df_name_scores_50[df_name_scores_50['name'].isin(names_50)]

df_names_50 = df_names_50.merge(datadf_pos[['Alert 1','Alert 2', 'Alert 3', 'Alert 4', 'Alert 5', 'Alert 6', 
                    'Alert 7','Alert 8', 'Alert 9', 'Alert 10', 'Alert 11', 'Alert 12','name']],on="name")

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, rdDepictor
from rdkit.Chem import AllChem
import numpy as np
rdDepictor.SetPreferCoordGen(True)

def _mol_from_smiles_with_coords(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles!r}")
    mol = Chem.AddHs(mol)  # helps coord generation; we won't draw Hs later
    AllChem.Compute2DCoords(mol)
    mol = Chem.RemoveHs(mol)  # draw without Hs by default
    return mol

def _bonds_between_atoms(mol: Chem.Mol, atom_set: set):
    """Return indices of bonds whose both endpoints are in atom_set."""
    bond_ids = []
    for b in mol.GetBonds():
        if b.GetBeginAtomIdx() in atom_set and b.GetEndAtomIdx() in atom_set:
            bond_ids.append(b.GetIdx())
    return bond_ids

def _palette():
    # RDKit expects RGB in [0,1]; distinct, readable colors
    return [
        (0.90, 0.20, 0.20),  # red
        (0.20, 0.60, 1.00),  # blue
        (0.10, 0.70, 0.30),  # green
        (0.95, 0.60, 0.10),  # orange
        (0.60, 0.30, 0.80),  # purple
        (0.20, 0.80, 0.80),  # teal
        (0.85, 0.85, 0.20),  # yellow
        (0.80, 0.40, 0.40),  # brick
    ]


def draw_selected_nodes_and_bonds(name: str, smiles: str, node_scores, size=(500, 400)):
    """
    Plot the molecule with atoms (score=1) and bonds between them highlighted.
    Returns a PIL.Image.Image.
    """
    mol = _mol_from_smiles_with_coords(smiles)
    n_atoms = mol.GetNumAtoms()

    scores = np.asarray(node_scores).astype(int)
    if scores.shape[0] != n_atoms:
        raise ValueError(f"node_scores length {len(scores)} != number of atoms {n_atoms}")

    highlight_atoms = set(np.nonzero(scores)[0].tolist())
    highlight_bonds = _bonds_between_atoms(mol, highlight_atoms)

    # Single color for the selection
    sel_color = (0.20, 0.60, 1.00)  # blue-ish
    atom_colors = {idx: sel_color for idx in highlight_atoms}
    bond_colors = {idx: sel_color for idx in highlight_bonds}

    img = Draw.MolToImage(
        mol,
        size=size,
        legend=f"{name} — selected atoms",
        highlightAtoms=sorted(highlight_atoms),
        highlightBonds=sorted(highlight_bonds),
        highlightAtomColors=atom_colors,
        highlightBondColors=bond_colors,
    )
    return img

def draw_smarts_highlight(name: str, smiles: str, smarts_list, size=(500, 400), use_chirality=True):
    """
    Plot the molecule with each SMARTS substructure highlighted (different color per SMARTS).
    Returns a PIL.Image.Image.
    """
    mol = _mol_from_smiles_with_coords(smiles)
    pals = _palette()

    all_highlight_atoms = set()
    all_highlight_bonds = set()
    atom_colors = {}
    bond_colors = {}

    for k, smarts in enumerate(smarts_list):
        if not smarts:
            continue
        q = Chem.MolFromSmarts(smarts)
        if q is None:
            # skip invalid SMARTS
            continue

        matches = mol.GetSubstructMatches(q, useChirality=use_chirality)
        if not matches:
            continue

        color = pals[k % len(pals)]
        for match in matches:
            # match: tuple mapping query atom idx -> target atom idx
            match_atoms = set(match)
            all_highlight_atoms.update(match_atoms)
            # color atoms
            for a in match_atoms:
                atom_colors[a] = color

            # color bonds by mapping query bonds through the match
            for qb in q.GetBonds():
                qa = qb.GetBeginAtomIdx()
                qb_ = qb.GetEndAtomIdx()
                ta = match[qa]
                tb = match[qb_]
                bond = mol.GetBondBetweenAtoms(ta, tb)
                if bond is not None:
                    bidx = bond.GetIdx()
                    all_highlight_bonds.add(bidx)
                    bond_colors[bidx] = color

    img = Draw.MolToImage(
        mol,
        size=size,
        legend=f"{name} — SMARTS highlight",
        highlightAtoms=sorted(all_highlight_atoms),
        highlightBonds=sorted(all_highlight_bonds),
        highlightAtomColors=atom_colors,
        highlightBondColors=bond_colors,
    )
    return img

In [ ]:
for assay in ['Liver_2vs']:
    for xplain_methon in ['Layer_Saliency']:

        for index,row in df_names_25.iterrows():
                compund_name = row['name']
    
    
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_127 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_128 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_129 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_130 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_131 = pickle.load(fp)

                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_127 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_128 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_129 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_130 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_131 = pickle.load(fp)

                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_127 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_128 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_129 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_130 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_131 = pickle.load(fp)




                ## GNN encoder
                gnn_encoder_mean_node_scores = torch.mean(torch.concat((gnn_encoder_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)



                gnn_encoder_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)


                res_gnn_encoder = {'node_scores':gnn_encoder_mean_node_scores,
                                   'node_scores_10':obtain_topk(gnn_encoder_mean_node_scores,0.10),
                                    'node_scores_25':obtain_topk(gnn_encoder_mean_node_scores,0.25),
                                   'node_scores_50':obtain_topk(gnn_encoder_mean_node_scores,0.50),
                                   'bond_idx':gnn_encoder_explanation_127['bond_idx'],
                                    'edge_scores':gnn_encoder_mean_edge_scores, 
                                   'edge_scores_25':obtain_topk(gnn_encoder_mean_edge_scores,0.25),
                                   'edge_scores_50':obtain_topk(gnn_encoder_mean_edge_scores,0.50)}


                ## GNN encoder + sem
                gnn_encoder_sem_mean_node_scores = torch.mean(torch.concat((gnn_encoder_sem_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)

                gnn_encoder_sem_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_sem_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)


                res_gnn_encoder_sem = {'node_scores':gnn_encoder_sem_mean_node_scores,
                                       'node_scores_10':obtain_topk(gnn_encoder_sem_mean_node_scores,0.10),
                                    'node_scores_25':obtain_topk(gnn_encoder_sem_mean_node_scores,0.25),
                                       'node_scores_50':obtain_topk(gnn_encoder_sem_mean_node_scores,0.50),
                                   'atom_idx_in_hetero':gnn_encoder_sem_explanation_129['atom_idx_in_hetero'],
                                    'edge_scores_25':gnn_encoder_sem_mean_edge_scores,
                                   'edge_scores_25':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.25),
                                    'edge_scores_50':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.50),
                                   'bond_idx_in_hetero':gnn_encoder_sem_explanation_129['bond_idx_in_hetero']}

                ## GNN encoder + sem CL
                gnn_encoder_semCL_mean_node_scores = torch.mean(torch.concat((gnn_encoder_semCL_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)

                gnn_encoder_semCL_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_semCL_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)

                res_gnn_encoder_semCL = {'node_scores':gnn_encoder_semCL_mean_node_scores,
                                         'node_scores_10':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.10),
                                        'node_scores_25':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.25),
                                         'node_scores_50':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.50),
                                   'atom_idx_in_hetero':gnn_encoder_semCL_explanation_127['atom_idx_in_hetero'],
                                'edge_scores':gnn_encoder_sem_mean_edge_scores,
                                   'edge_scores_25':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.25),
                                    'edge_scores_50':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.50),
                                   'bond_idx_in_hetero':gnn_encoder_semCL_explanation_127['bond_idx_in_hetero']}
    
                ###
                tmp_mol = _mol_from_smiles_with_coords(row['smiles'])
                node_scores_encoder = res_gnn_encoder['node_scores_25']
                node_scores_semantic = res_gnn_encoder_sem['node_scores_25']
                node_scores_semantic_CL = res_gnn_encoder_semCL['node_scores_25']
                
                smarts_list = []
                for alert in ['Alert 1','Alert 2', 'Alert 3', 'Alert 4', 'Alert 5', 'Alert 6', 
                    'Alert 7','Alert 8', 'Alert 9', 'Alert 10', 'Alert 11', 'Alert 12']:
                    if row[alert]==1:
                        smarts_list.append(Alert_dict[alert])
                        
                img_gnn_encoder = draw_selected_nodes_and_bonds(row['name'], row['smiles'], node_scores_encoder)
                img_gnn_semantic = draw_selected_nodes_and_bonds(row['name'], row['smiles'], node_scores_semantic)
                img_gnn_semantic_CL = draw_selected_nodes_and_bonds(row['name'], row['smiles'], node_scores_semantic_CL)
                img_gnn_true = draw_smarts_highlight(row['name'], row['smiles'], smarts_list)
                
                
                img_gnn_encoder.save("./MolCLR/results_liver/explain/25img_gnn_encoder" + row['name'] + ".png")
                img_gnn_semantic.save("./MolCLR/results_liver/explain_sem/25img_gnn_semantic" + row['name'] + ".png")
                img_gnn_semantic_CL.save("./MolCLR/results_liver/explain_sem_CL/25img_gnn_semantic_CL" + row['name'] + ".png")
                img_gnn_true.save("./MolCLR/results_liver/explain/25img_gnn_true" + row['name'] + ".png")

                
                
################

for assay in ['Liver_2vs']:
    for xplain_methon in ['Layer_Saliency']:

        for index,row in df_names_50.iterrows():
                compund_name = row['name']
    
    
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_127 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_128 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_129 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_130 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_131 = pickle.load(fp)

                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_127 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_128 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_129 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_130 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_131 = pickle.load(fp)

                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_127 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_128 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_129 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_130 = pickle.load(fp)
                with open("./MolCLR/results_Liver/explain_sem_CL/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_131 = pickle.load(fp)




                ## GNN encoder
                gnn_encoder_mean_node_scores = torch.mean(torch.concat((gnn_encoder_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)



                gnn_encoder_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)


                res_gnn_encoder = {'node_scores':gnn_encoder_mean_node_scores,
                                   'node_scores_10':obtain_topk(gnn_encoder_mean_node_scores,0.10),
                                    'node_scores_25':obtain_topk(gnn_encoder_mean_node_scores,0.25),
                                   'node_scores_50':obtain_topk(gnn_encoder_mean_node_scores,0.50),
                                   'bond_idx':gnn_encoder_explanation_127['bond_idx'],
                                    'edge_scores':gnn_encoder_mean_edge_scores, 
                                   'edge_scores_25':obtain_topk(gnn_encoder_mean_edge_scores,0.25),
                                   'edge_scores_50':obtain_topk(gnn_encoder_mean_edge_scores,0.50)}


                ## GNN encoder + sem
                gnn_encoder_sem_mean_node_scores = torch.mean(torch.concat((gnn_encoder_sem_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)

                gnn_encoder_sem_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_sem_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)


                res_gnn_encoder_sem = {'node_scores':gnn_encoder_sem_mean_node_scores,
                                       'node_scores_10':obtain_topk(gnn_encoder_sem_mean_node_scores,0.10),
                                    'node_scores_25':obtain_topk(gnn_encoder_sem_mean_node_scores,0.25),
                                       'node_scores_50':obtain_topk(gnn_encoder_sem_mean_node_scores,0.50),
                                   'atom_idx_in_hetero':gnn_encoder_sem_explanation_129['atom_idx_in_hetero'],
                                    'edge_scores_25':gnn_encoder_sem_mean_edge_scores,
                                   'edge_scores_25':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.25),
                                    'edge_scores_50':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.50),
                                   'bond_idx_in_hetero':gnn_encoder_sem_explanation_129['bond_idx_in_hetero']}

                ## GNN encoder + sem CL
                gnn_encoder_semCL_mean_node_scores = torch.mean(torch.concat((gnn_encoder_semCL_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)

                gnn_encoder_semCL_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_semCL_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)

                res_gnn_encoder_semCL = {'node_scores':gnn_encoder_semCL_mean_node_scores,
                                         'node_scores_10':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.10),
                                        'node_scores_25':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.25),
                                         'node_scores_50':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.50),
                                   'atom_idx_in_hetero':gnn_encoder_semCL_explanation_127['atom_idx_in_hetero'],
                                'edge_scores':gnn_encoder_sem_mean_edge_scores,
                                   'edge_scores_25':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.25),
                                    'edge_scores_50':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.50),
                                   'bond_idx_in_hetero':gnn_encoder_semCL_explanation_127['bond_idx_in_hetero']}
    
                ###
                tmp_mol = _mol_from_smiles_with_coords(row['smiles'])
                node_scores_encoder = res_gnn_encoder['node_scores_50']
                node_scores_semantic = res_gnn_encoder_sem['node_scores_50']
                node_scores_semantic_CL = res_gnn_encoder_semCL['node_scores_50']
                
                smarts_list = []
                for alert in ['Alert 1','Alert 2', 'Alert 3', 'Alert 4', 'Alert 5', 'Alert 6', 
                    'Alert 7','Alert 8', 'Alert 9', 'Alert 10', 'Alert 11', 'Alert 12']:
                    if row[alert]==1:
                        smarts_list.append(Alert_dict[alert])
                        
                img_gnn_encoder = draw_selected_nodes_and_bonds(row['name'], row['smiles'], node_scores_encoder)
                img_gnn_semantic = draw_selected_nodes_and_bonds(row['name'], row['smiles'], node_scores_semantic)
                img_gnn_semantic_CL = draw_selected_nodes_and_bonds(row['name'], row['smiles'], node_scores_semantic_CL)
                img_gnn_true = draw_smarts_highlight(row['name'], row['smiles'], smarts_list)
                
                
                img_gnn_encoder.save("./MolCLR/results_liver/explain/50img_gnn_encoder" + row['name'] + ".png")
                img_gnn_semantic.save("./MolCLR/results_liver/explain_sem/50img_gnn_semantic" + row['name'] + ".png")
                img_gnn_semantic_CL.save("./MolCLR/results_liver/explain_sem_CL/50img_gnn_semantic_CL" + row['name'] + ".png")
                img_gnn_true.save("./MolCLR/results_liver/explain/50img_gnn_true" + row['name'] + ".png")

                

In [ ]:
    print("Accuracy")
    print("GNN encoder")
    print("top-10% - " + str(np.mean(acc_gnn_encoder_10)))
    print("top-25% - " + str(np.mean(acc_gnn_encoder_25)))
    print("top-50% - " + str(np.mean(acc_gnn_encoder_50)))
    
    print("GNN encoder + sem")
    print("top-10% - " + str(np.mean(acc_gnn_encoder_sem_10)))
    print("top-25% - " + str(np.mean(acc_gnn_encoder_sem_25)))
    print("top-50% - " + str(np.mean(acc_gnn_encoder_sem_50)))
        
    print("GNN encoder + sem CL")
    print("top-10% - " + str(np.mean(acc_gnn_encoder_semCL_10)))
    print("top-25% - " + str(np.mean(acc_gnn_encoder_semCL_25)))
    print("top-50% - " + str(np.mean(acc_gnn_encoder_semCL_50)))
    
    df_plot_acc = pd.DataFrame({"GNN encoder - top-25%":acc_gnn_encoder_25,
                             "GNN encoder - top-50%":acc_gnn_encoder_50,
                             "GNN encoder + sem - top-25%":acc_gnn_encoder_sem_25,
                             "GNN encoder + sem - top-50%":acc_gnn_encoder_sem_50,
                             "GNN encoder + sem CL - top-25%":acc_gnn_encoder_semCL_25,
                             "GNN encoder + sem CL - top-50%":acc_gnn_encoder_semCL_50})
    
    print("\n\n")
    print("Recall - sens")
    print("GNN encoder")
    print("top-10% - " + str(np.mean(rec_gnn_encoder_10)))
    print("top-25% - " + str(np.mean(rec_gnn_encoder_25)))
    print("top-50% - " + str(np.mean(rec_gnn_encoder_50)))
    
    print("GNN encoder + sem")
    print("top-10% - " + str(np.mean(rec_gnn_encoder_sem_10)))
    print("top-25% - " + str(np.mean(rec_gnn_encoder_sem_25)))
    print("top-50% - " + str(np.mean(rec_gnn_encoder_sem_50)))
        
    print("GNN encoder + sem CL")
    print("top-10% - " + str(np.mean(rec_gnn_encoder_semCL_10)))
    print("top-25% - " + str(np.mean(rec_gnn_encoder_semCL_25)))
    print("top-50% - " + str(np.mean(rec_gnn_encoder_semCL_50)))
    
    df_plot_rec = pd.DataFrame({"GNN encoder - top-25%":rec_gnn_encoder_25,
                             "GNN encoder - top-50%":rec_gnn_encoder_50,
                             "GNN encoder + sem - top-25%":rec_gnn_encoder_sem_25,
                             "GNN encoder + sem - top-50%":rec_gnn_encoder_sem_50,
                             "GNN encoder + sem CL - top-25%":rec_gnn_encoder_semCL_25,
                             "GNN encoder + sem CL - top-50%":rec_gnn_encoder_semCL_50})
    
    print("\n\n")
    print("Precision - ppv")
    print("GNN encoder")
    print("top-10% - " + str(np.mean(pre_gnn_encoder_10)))
    print("top-25% - " + str(np.mean(pre_gnn_encoder_25)))
    print("top-50% - " + str(np.mean(pre_gnn_encoder_50)))
    
    print("GNN encoder + sem")
    print("top-10% - " + str(np.mean(pre_gnn_encoder_sem_10)))
    print("top-25% - " + str(np.mean(pre_gnn_encoder_sem_25)))
    print("top-50% - " + str(np.mean(pre_gnn_encoder_sem_50)))
        
    print("GNN encoder + sem CL")
    print("top-10% - " + str(np.mean(pre_gnn_encoder_semCL_10)))
    print("top-25% - " + str(np.mean(pre_gnn_encoder_semCL_25)))
    print("top-50% - " + str(np.mean(pre_gnn_encoder_semCL_50)))
    
    df_plot_pre = pd.DataFrame({"GNN encoder - top-25%":pre_gnn_encoder_25,
                             "GNN encoder - top-50%":pre_gnn_encoder_50,
                             "GNN encoder + sem - top-25%":pre_gnn_encoder_sem_25,
                             "GNN encoder + sem - top-50%":pre_gnn_encoder_sem_50,
                             "GNN encoder + sem CL - top-25%":pre_gnn_encoder_semCL_25,
                             "GNN encoder + sem CL - top-50%":pre_gnn_encoder_semCL_50})
    
    print("\n\n")
    print("AUC")
    print("GNN encoder")
    print("top-10% - " + str(np.mean(auc_gnn_encoder_10)))
    print("top-25% - " + str(np.mean(auc_gnn_encoder_25)))
    print("top-50% - " + str(np.mean(auc_gnn_encoder_50)))
    print("prob " + str(np.mean(auc_gnn_encoder)))
    
    print("GNN encoder + sem")
    print("top-10% - " + str(np.mean(auc_gnn_encoder_sem_10)))
    print("top-25% - " + str(np.mean(auc_gnn_encoder_sem_25)))
    print("top-50% - " + str(np.mean(auc_gnn_encoder_sem_50)))
    print("prob " + str(np.mean(auc_gnn_encoder_sem)))
        
    print("GNN encoder + sem CL")
    print("top-10% - " + str(np.mean(auc_gnn_encoder_semCL_10)))
    print("top-25% - " + str(np.mean(auc_gnn_encoder_semCL_25)))
    print("top-50% - " + str(np.mean(auc_gnn_encoder_semCL_50)))
    print("prob " + str(np.mean(auc_gnn_encoder_semCL_50)))
    
    df_plot_auc = pd.DataFrame({"GNN encoder - top-25%":auc_gnn_encoder_25,
                             "GNN encoder - top-50%":auc_gnn_encoder_50,
                            "GNN encoder - prob":auc_gnn_encoder,
                             "GNN encoder + sem - top-25%":auc_gnn_encoder_sem_25,
                             "GNN encoder + sem - top-50%":auc_gnn_encoder_sem_50,
                                 "GNN encoder + sem - prob":auc_gnn_encoder_sem,
                             "GNN encoder + sem CL - top-25%":auc_gnn_encoder_semCL_25,
                             "GNN encoder + sem CL - top-50%":auc_gnn_encoder_semCL_50,
                               "GNN encoder + sem CL - prob":auc_gnn_encoder_semCL})

In [ ]:




Alert_dict = {'Alert 1':"C12CCCCC1C3C(CCC3)CC2", 
              'Alert 2':"NN", 
              'Alert 3':"a[C!R]C(=O)[OH]",
              'Alert 4':"[#6]S(=O)(=O)N[#6]", 
              'Alert 5':"c1ccccc1[NH2]", 
              'Alert 6':"O = [S;X3]", 
              'Alert 7':"[S;X2&!R]", 
              'Alert 8':"a[C!R](=O)a", 
              'Alert 9':"C[F,Cl,Br,I]", 
              'Alert 10':"C1CC1N", 
              'Alert 11':"[O]c1ccc([N])cc1", 
              'Alert 12':"N1c2ccccc2Sc2ccccc12"}


acc_gnn_encoder_10 = []
acc_gnn_encoder_25 = []
acc_gnn_encoder_50 = []
acc_gnn_encoder_sem_10 = []
acc_gnn_encoder_sem_25 = []
acc_gnn_encoder_sem_50 = []
acc_gnn_encoder_semCL_10 = []
acc_gnn_encoder_semCL_25 = []
acc_gnn_encoder_semCL_50 = []

pre_gnn_encoder_10 = []
pre_gnn_encoder_25 = []
pre_gnn_encoder_50 = []
pre_gnn_encoder_sem_10 = []
pre_gnn_encoder_sem_25 = []
pre_gnn_encoder_sem_50 = []
pre_gnn_encoder_semCL_10 = []
pre_gnn_encoder_semCL_25 = []
pre_gnn_encoder_semCL_50 = []

rec_gnn_encoder_10 = []
rec_gnn_encoder_25 = []
rec_gnn_encoder_50 = []
rec_gnn_encoder_sem_10 = []
rec_gnn_encoder_sem_25 = []
rec_gnn_encoder_sem_50 = []
rec_gnn_encoder_semCL_10 = []
rec_gnn_encoder_semCL_25 = []
rec_gnn_encoder_semCL_50 = []

auc_gnn_encoder_10 = []
auc_gnn_encoder_25 = []
auc_gnn_encoder_50 = []
auc_gnn_encoder_sem_10 = []
auc_gnn_encoder_sem_25 = []
auc_gnn_encoder_sem_50 = []
auc_gnn_encoder_semCL_10 = []
auc_gnn_encoder_semCL_25 = []
auc_gnn_encoder_semCL_50 = []

auc_gnn_encoder = []
auc_gnn_encoder_sem = []
auc_gnn_encoder_semCL = []

for assay in ['Liver_2vs']:
    
    datadf = pd.read_excel("./MolCLR/data/Liver/datasets_valid_and_splits/" + assay + "_df.xlsx")
    datadf_pos = datadf[datadf['target'] == 1]

    
    attribution_df = pd.read_excel("./MolCLR/data/Liver/Liver_.xlsx")
    attribution_df['name'] = attribution_df['Drug Name']
    attribution_df['smiles'] = attribution_df['n.sMILES']
    
    datadf_pos = datadf_pos.merge(attribution_df[['name','smiles','attribution','Alert 1', 'Alert 2', 'Alert 3', 'Alert 4', 'Alert 5', 'Alert 6', 'Alert 7', 'Alert 8', 'Alert 9', 'Alert 10', 'Alert 11', 'Alert 12']],on=['name','smiles'])
    
    
    for xplain_methon in ['Layer_Saliency']:
        for index,row in datadf_pos.iterrows():
            
            if row[['Alert 1','Alert 2', 'Alert 3', 'Alert 4', 'Alert 5', 'Alert 6', 
                    'Alert 7','Alert 8', 'Alert 9', 'Alert 10', 'Alert 11', 'Alert 12']].values.sum() != 0:
            
                compund_name = row['name']

                with open("./HiMol/finetune/results_Liver/explain/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_127 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_128 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_129 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_130 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_explanation_131 = pickle.load(fp)

                with open("./HiMol/finetune/results_Liver/explain_sem/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_127 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain_sem/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_128 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain_sem/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_129 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain_sem/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_130 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain_sem/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_sem_explanation_131 = pickle.load(fp)

                with open("./HiMol/finetune/results_Liver/explain_sem_CL/" + xplain_methon + "/127/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_127 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain_sem_CL/" + xplain_methon + "/128/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_128 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain_sem_CL/" + xplain_methon + "/129/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_129 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain_sem_CL/" + xplain_methon + "/130/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_130 = pickle.load(fp)
                with open("./HiMol/finetune/results_Liver/explain_sem_CL/" + xplain_methon + "/131/" + assay + "/" + compund_name + ".pkl", "rb") as fp:
                    gnn_encoder_semCL_explanation_131 = pickle.load(fp)




                ## GNN encoder
                gnn_encoder_mean_node_scores = torch.mean(torch.concat((gnn_encoder_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)



                gnn_encoder_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)


                res_gnn_encoder = {'node_scores':gnn_encoder_mean_node_scores,
                                   'node_scores_10':obtain_topk(gnn_encoder_mean_node_scores,0.10),
                                    'node_scores_25':obtain_topk(gnn_encoder_mean_node_scores,0.25),
                                   'node_scores_50':obtain_topk(gnn_encoder_mean_node_scores,0.50),
                                   'bond_idx':gnn_encoder_explanation_127['bond_idx'],
                                    'edge_scores':gnn_encoder_mean_edge_scores, 
                                   'edge_scores_25':obtain_topk(gnn_encoder_mean_edge_scores,0.25),
                                   'edge_scores_50':obtain_topk(gnn_encoder_mean_edge_scores,0.50)}


                ## GNN encoder + sem
                gnn_encoder_sem_mean_node_scores = torch.mean(torch.concat((gnn_encoder_sem_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)

                gnn_encoder_sem_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_sem_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_sem_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)


                res_gnn_encoder_sem = {'node_scores':gnn_encoder_sem_mean_node_scores,
                                       'node_scores_10':obtain_topk(gnn_encoder_sem_mean_node_scores,0.10),
                                    'node_scores_25':obtain_topk(gnn_encoder_sem_mean_node_scores,0.25),
                                       'node_scores_50':obtain_topk(gnn_encoder_sem_mean_node_scores,0.50),
                                   'atom_idx_in_hetero':gnn_encoder_sem_explanation_129['atom_idx_in_hetero'],
                                    'edge_scores_25':gnn_encoder_sem_mean_edge_scores,
                                   'edge_scores_25':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.25),
                                    'edge_scores_50':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.50),
                                   'bond_idx_in_hetero':gnn_encoder_sem_explanation_129['bond_idx_in_hetero']}

                ## GNN encoder + sem CL
                gnn_encoder_semCL_mean_node_scores = torch.mean(torch.concat((gnn_encoder_semCL_explanation_127['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_128['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_129['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_130['node_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_131['node_scores'].reshape(1,-1)),dim=0),dim=0)

                gnn_encoder_semCL_mean_edge_scores = torch.mean(torch.concat((gnn_encoder_semCL_explanation_127['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_128['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_129['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_130['edge_scores'].reshape(1,-1),
                                                            gnn_encoder_semCL_explanation_131['edge_scores'].reshape(1,-1)),dim=0),dim=0)

                res_gnn_encoder_semCL = {'node_scores':gnn_encoder_semCL_mean_node_scores,
                                         'node_scores_10':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.10),
                                        'node_scores_25':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.25),
                                         'node_scores_50':obtain_topk(gnn_encoder_semCL_mean_node_scores,0.50),
                                   'atom_idx_in_hetero':gnn_encoder_semCL_explanation_127['atom_idx_in_hetero'],
                                'edge_scores':gnn_encoder_sem_mean_edge_scores,
                                   'edge_scores_25':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.25),
                                    'edge_scores_50':obtain_topk(gnn_encoder_sem_mean_edge_scores,0.50),
                                   'bond_idx_in_hetero':gnn_encoder_semCL_explanation_127['bond_idx_in_hetero']}
                
                acc_gnn_encoder_10.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_10'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_25.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_25'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_50.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_50'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_sem_10.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_10'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_sem_25.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_25'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_sem_50.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_50'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_semCL_10.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_10'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_semCL_25.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_25'], Alert_dict, n_alerts=12)['accuracy'])
                acc_gnn_encoder_semCL_50.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_50'], Alert_dict, n_alerts=12)['accuracy'])

                pre_gnn_encoder_10.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_10'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_25.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_25'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_50.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_50'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_sem_10.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_10'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_sem_25.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_25'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_sem_50.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_50'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_semCL_10.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_10'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_semCL_25.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_25'], Alert_dict, n_alerts=12)['precision'])
                pre_gnn_encoder_semCL_50.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_50'], Alert_dict, n_alerts=12)['precision'])

                
                rec_gnn_encoder_10.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_10'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_25.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_25'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_50.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_50'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_sem_10.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_10'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_sem_25.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_25'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_sem_50.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_50'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_semCL_10.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_10'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_semCL_25.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_25'], Alert_dict, n_alerts=12)['recall'])
                rec_gnn_encoder_semCL_50.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_50'], Alert_dict, n_alerts=12)['recall'])

                
                auc_gnn_encoder_10.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_10'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_25.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_25'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_50.append(node_attribution_metrics(row, res_gnn_encoder['node_scores_50'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_sem_10.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_10'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_sem_25.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_25'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_sem_50.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores_50'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_semCL_10.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_10'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_semCL_25.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_25'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_semCL_50.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores_50'], Alert_dict, n_alerts=12)['auroc'])

                auc_gnn_encoder.append(node_attribution_metrics(row, res_gnn_encoder['node_scores'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_sem.append(node_attribution_metrics(row, res_gnn_encoder_sem['node_scores'], Alert_dict, n_alerts=12)['auroc'])
                auc_gnn_encoder_semCL.append(node_attribution_metrics(row, res_gnn_encoder_semCL['node_scores'], Alert_dict, n_alerts=12)['auroc'])
                
         

In [ ]:
    print("Accuracy")
    print("GNN encoder")
    print("top-10% - " + str(np.mean(acc_gnn_encoder_10)))
    print("top-25% - " + str(np.mean(acc_gnn_encoder_25)))
    print("top-50% - " + str(np.mean(acc_gnn_encoder_50)))
    
    print("GNN encoder + sem")
    print("top-10% - " + str(np.mean(acc_gnn_encoder_sem_10)))
    print("top-25% - " + str(np.mean(acc_gnn_encoder_sem_25)))
    print("top-50% - " + str(np.mean(acc_gnn_encoder_sem_50)))
        
    print("GNN encoder + sem CL")
    print("top-10% - " + str(np.mean(acc_gnn_encoder_semCL_10)))
    print("top-25% - " + str(np.mean(acc_gnn_encoder_semCL_25)))
    print("top-50% - " + str(np.mean(acc_gnn_encoder_semCL_50)))
    
    df_plot_acc = pd.DataFrame({"GNN encoder - top-25%":acc_gnn_encoder_25,
                             "GNN encoder - top-50%":acc_gnn_encoder_50,
                             "GNN encoder + sem - top-25%":acc_gnn_encoder_sem_25,
                             "GNN encoder + sem - top-50%":acc_gnn_encoder_sem_50,
                             "GNN encoder + sem CL - top-25%":acc_gnn_encoder_semCL_25,
                             "GNN encoder + sem CL - top-50%":acc_gnn_encoder_semCL_50})
    
    print("\n\n")
    print("Recall - sens")
    print("GNN encoder")
    print("top-10% - " + str(np.mean(rec_gnn_encoder_10)))
    print("top-25% - " + str(np.mean(rec_gnn_encoder_25)))
    print("top-50% - " + str(np.mean(rec_gnn_encoder_50)))
    
    print("GNN encoder + sem")
    print("top-10% - " + str(np.mean(rec_gnn_encoder_sem_10)))
    print("top-25% - " + str(np.mean(rec_gnn_encoder_sem_25)))
    print("top-50% - " + str(np.mean(rec_gnn_encoder_sem_50)))
        
    print("GNN encoder + sem CL")
    print("top-10% - " + str(np.mean(rec_gnn_encoder_semCL_10)))
    print("top-25% - " + str(np.mean(rec_gnn_encoder_semCL_25)))
    print("top-50% - " + str(np.mean(rec_gnn_encoder_semCL_50)))
    
    df_plot_rec = pd.DataFrame({"GNN encoder - top-25%":rec_gnn_encoder_25,
                             "GNN encoder - top-50%":rec_gnn_encoder_50,
                             "GNN encoder + sem - top-25%":rec_gnn_encoder_sem_25,
                             "GNN encoder + sem - top-50%":rec_gnn_encoder_sem_50,
                             "GNN encoder + sem CL - top-25%":rec_gnn_encoder_semCL_25,
                             "GNN encoder + sem CL - top-50%":rec_gnn_encoder_semCL_50})
    
    print("\n\n")
    print("Precision - ppv")
    print("GNN encoder")
    print("top-10% - " + str(np.mean(pre_gnn_encoder_10)))
    print("top-25% - " + str(np.mean(pre_gnn_encoder_25)))
    print("top-50% - " + str(np.mean(pre_gnn_encoder_50)))
    
    print("GNN encoder + sem")
    print("top-10% - " + str(np.mean(pre_gnn_encoder_sem_10)))
    print("top-25% - " + str(np.mean(pre_gnn_encoder_sem_25)))
    print("top-50% - " + str(np.mean(pre_gnn_encoder_sem_50)))
        
    print("GNN encoder + sem CL")
    print("top-10% - " + str(np.mean(pre_gnn_encoder_semCL_10)))
    print("top-25% - " + str(np.mean(pre_gnn_encoder_semCL_25)))
    print("top-50% - " + str(np.mean(pre_gnn_encoder_semCL_50)))
    
    df_plot_pre = pd.DataFrame({"GNN encoder - top-25%":pre_gnn_encoder_25,
                             "GNN encoder - top-50%":pre_gnn_encoder_50,
                             "GNN encoder + sem - top-25%":pre_gnn_encoder_sem_25,
                             "GNN encoder + sem - top-50%":pre_gnn_encoder_sem_50,
                             "GNN encoder + sem CL - top-25%":pre_gnn_encoder_semCL_25,
                             "GNN encoder + sem CL - top-50%":pre_gnn_encoder_semCL_50})
    
    print("\n\n")
    print("AUC")
    print("GNN encoder")
    print("top-10% - " + str(np.mean(auc_gnn_encoder_10)))
    print("top-25% - " + str(np.mean(auc_gnn_encoder_25)))
    print("top-50% - " + str(np.mean(auc_gnn_encoder_50)))
    print("prob " + str(np.mean(auc_gnn_encoder)))
    
    print("GNN encoder + sem")
    print("top-10% - " + str(np.mean(auc_gnn_encoder_sem_10)))
    print("top-25% - " + str(np.mean(auc_gnn_encoder_sem_25)))
    print("top-50% - " + str(np.mean(auc_gnn_encoder_sem_50)))
    print("prob " + str(np.mean(auc_gnn_encoder_sem)))
        
    print("GNN encoder + sem CL")
    print("top-10% - " + str(np.mean(auc_gnn_encoder_semCL_10)))
    print("top-25% - " + str(np.mean(auc_gnn_encoder_semCL_25)))
    print("top-50% - " + str(np.mean(auc_gnn_encoder_semCL_50)))
    print("prob " + str(np.mean(auc_gnn_encoder_semCL_50)))
    
    df_plot_auc = pd.DataFrame({"GNN encoder - top-25%":auc_gnn_encoder_25,
                             "GNN encoder - top-50%":auc_gnn_encoder_50,
                            "GNN encoder - prob":auc_gnn_encoder,
                             "GNN encoder + sem - top-25%":auc_gnn_encoder_sem_25,
                             "GNN encoder + sem - top-50%":auc_gnn_encoder_sem_50,
                                 "GNN encoder + sem - prob":auc_gnn_encoder_sem,
                             "GNN encoder + sem CL - top-25%":auc_gnn_encoder_semCL_25,
                             "GNN encoder + sem CL - top-50%":auc_gnn_encoder_semCL_50,
                               "GNN encoder + sem CL - prob":auc_gnn_encoder_semCL})